# Issue #15 — Validate Databento Data and Contract Mapping

Verify downloaded data quality and roll-pair construction before downstream analysis.


In [1]:
import pandas as pd
import numpy as np
import re

ohlcv = pd.read_parquet('../data/raw/zn_ohlcv_daily.parquet')
oi    = pd.read_parquet('../data/raw/zn_open_interest.parquet')
rolls = pd.read_parquet('../data/raw/zn_roll_pairs.parquet')
panel = pd.read_parquet('../data/processed/zn_roll_panel.parquet')
cross = pd.read_parquet('../data/processed/zn_crossovers.parquet')

ohlcv['date'] = pd.to_datetime(ohlcv['date'])
oi['date']    = pd.to_datetime(oi['date'])
rolls['fnd']  = pd.to_datetime(rolls['fnd'])
panel['date'] = pd.to_datetime(panel['date'])
if 'crossover_date' in cross.columns:
    cross['crossover_date'] = pd.to_datetime(cross['crossover_date'])

print('Data loaded.')


Data loaded.


## 1. Check contract symbols match expected pattern


In [2]:
outright_pat = re.compile(r'^ZN[HMUZ]\d$')
ohlcv_syms = ohlcv['symbol'].unique()
outrights = [s for s in ohlcv_syms if outright_pat.match(s)]
spreads   = [s for s in ohlcv_syms if '-' in s and s.startswith('ZN')]
other     = [s for s in ohlcv_syms if s not in outrights and s not in spreads]

checks = []
checks.append({'Check': 'Outright symbols match ZN[HMUZ]\d', 'Count': len(outrights), 'Status': '✅'})
checks.append({'Check': 'Spread symbols contain "-"', 'Count': len(spreads), 'Status': '✅'})
checks.append({'Check': 'Unrecognized symbols', 'Count': len(other),
               'Status': '✅' if len(other) == 0 else '⚠️ ' + str(other[:5])})
pd.DataFrame(checks).set_index('Check')


,Count,Status
Check,,
Outright symbols match ZN[HMUZ]\d,29,✅
"Spread symbols contain ""-""",55,✅
Unrecognized symbols,753,"⚠️ ['UD:ZN: TL 0128820945', 'UD:ZN: TL 0128816..."


## 2. Verify front and next contract assignments


In [3]:
NEXT_MAP = {'H': 'M', 'M': 'U', 'U': 'Z', 'Z': 'H'}
bad_pairs = []
for _, r in rolls.iterrows():
    fc, nc = r['front'][2], r['next'][2]  # month codes
    expected_next = NEXT_MAP[fc]
    if nc != expected_next:
        bad_pairs.append(f'{r["front"]} → {r["next"]} (expected {expected_next})')

checks2 = []
checks2.append({'Check': 'Total roll pairs', 'Value': str(len(rolls))})
checks2.append({'Check': 'Pairs with correct front→next sequence', 'Value': str(len(rolls) - len(bad_pairs))})
checks2.append({'Check': 'Pairs with wrong sequence', 'Value': str(len(bad_pairs)) + (' ✅' if len(bad_pairs)==0 else ' ⚠️')})
pd.DataFrame(checks2).set_index('Check')


,Value
Check,
Total roll pairs,28
Pairs with correct front→next sequence,28
Pairs with wrong sequence,0 ✅


## 3. Validate First Notice Day calculations


In [4]:
# FND should be last business day of the month before the delivery month
# and should always fall on a weekday
fnd_checks = []
for _, r in rolls.iterrows():
    fnd = pd.Timestamp(r['fnd'])
    dm = r['delivery_month']
    # FND month should be dm - 1 (or 12 if dm is 1)
    expected_month = dm - 1 if dm > 1 else 12
    month_ok = fnd.month == expected_month
    weekday_ok = fnd.weekday() < 5
    # Should be last bday of that month (next day should be in a different month or weekend)
    next_day = fnd + pd.Timedelta(days=1)
    while next_day.weekday() >= 5:
        next_day += pd.Timedelta(days=1)
    last_bday_ok = next_day.month != fnd.month

    fnd_checks.append({
        'Roll': f'{r["front"]}→{r["next"]}',
        'FND': fnd.date(),
        'Month OK': '✅' if month_ok else '❌',
        'Weekday': '✅' if weekday_ok else '❌',
        'Last Bday': '✅' if last_bday_ok else '❌',
    })

fnd_df = pd.DataFrame(fnd_checks).set_index('Roll')
n_issues = ((fnd_df == '❌').any(axis=1)).sum()
print(f'FND validation: {len(fnd_df)} rolls checked, {n_issues} issues found')
fnd_df.head(10)


FND validation: 28 rolls checked, 0 issues found


,FND,Month OK,Weekday,Last Bday
Roll,,,,
ZNH0→ZNM0,2020-02-28,✅,✅,✅
ZNM0→ZNU0,2020-05-29,✅,✅,✅
ZNU0→ZNZ0,2020-08-31,✅,✅,✅
ZNZ0→ZNH1,2020-11-30,✅,✅,✅
ZNH1→ZNM1,2021-02-26,✅,✅,✅
ZNM1→ZNU1,2021-05-31,✅,✅,✅
ZNU1→ZNZ1,2021-08-31,✅,✅,✅
ZNZ1→ZNH2,2021-11-30,✅,✅,✅
ZNH2→ZNM2,2022-02-28,✅,✅,✅


## 4. Check for missing or duplicate observations


In [5]:
# Check for duplicates in OHLCV
dupes_ohlcv = ohlcv.groupby(['symbol', 'date']).size()
dupes = dupes_ohlcv[dupes_ohlcv > 1]

# Check for gaps in trading days for active contracts
gap_report = []
for sym in sorted(rolls['front'].unique())[:5]:  # sample 5
    sym_data = ohlcv[ohlcv['symbol'] == sym].sort_values('date')
    if len(sym_data) < 2:
        continue
    date_diffs = sym_data['date'].diff().dt.days
    max_gap = date_diffs.max()
    gap_report.append({'Symbol': sym, 'Trading Days': len(sym_data),
                       'Max Gap (calendar days)': int(max_gap) if pd.notna(max_gap) else 0,
                       'Date Range': f'{sym_data["date"].min().date()} → {sym_data["date"].max().date()}'})

dup_checks = []
dup_checks.append({'Check': 'Duplicate (symbol, date) in OHLCV', 'Value': str(len(dupes)) + (' ✅' if len(dupes)==0 else ' ⚠️')})
dup_checks.append({'Check': 'Duplicate (symbol, date) in OI', 'Value': str(len(oi.groupby(["symbol","date"]).size().pipe(lambda x: x[x>1]))) + ' ✅'})
display(pd.DataFrame(dup_checks).set_index('Check'))

print('\nSample contract coverage:')
pd.DataFrame(gap_report).set_index('Symbol')


,Value
Check,
"Duplicate (symbol, date) in OHLCV",0 ✅
"Duplicate (symbol, date) in OI",0 ✅



Sample contract coverage:


,Trading Days,Max Gap (calendar days),Date Range
Symbol,,,
ZNH0,68,3,2020-01-01 → 2020-03-20
ZNH1,141,8,2020-09-23 → 2021-03-22
ZNH2,164,13,2021-08-17 → 2022-03-22
ZNH3,169,19,2022-07-29 → 2023-03-22
ZNH4,171,16,2023-07-06 → 2024-03-19


## 5. Verify crossover dates are reasonable


In [6]:
if len(cross) > 0:
    cross_checks = []
    cross_checks.append({'Check': 'Rolls with identified crossover',
                         'Value': f'{len(cross)} / {len(rolls)}'})
    cross_checks.append({'Check': 'Median crossover (bdays to FND)',
                         'Value': f'{cross["crossover_bdays_to_fnd"].median():.0f}'})
    cross_checks.append({'Check': 'Range',
                         'Value': f'{cross["crossover_bdays_to_fnd"].min():.0f} to {cross["crossover_bdays_to_fnd"].max():.0f}'})
    cross_checks.append({'Check': 'Any crossover AFTER FND?',
                         'Value': ('⚠️ Yes' if (cross['crossover_bdays_to_fnd'] > 0).any() else '✅ No')})
    pd.DataFrame(cross_checks).set_index('Check')
else:
    print('No crossovers to validate (run notebook 02 first)')


## 6. Data quality summary


In [7]:
print('=' * 50)
print('DATA QUALITY SUMMARY')
print('=' * 50)
print(f'Date range:        {ohlcv["date"].min().date()} → {ohlcv["date"].max().date()}')
print(f'Outright contracts: {len(outrights)}')
print(f'Spread instruments: {len(spreads)}')
print(f'Roll pairs:         {len(rolls)}')
print(f'OHLCV rows:         {len(ohlcv):,}')
print(f'OI rows:            {len(oi):,}')
print(f'Panel rows:         {len(panel):,}')
print(f'Crossovers found:   {len(cross)}')
print(f'Symbol issues:      {len(other)}')
print(f'Pair seq issues:    {len(bad_pairs)}')
print(f'FND issues:         {n_issues}')
print(f'OHLCV duplicates:   {len(dupes)}')
print('=' * 50)
if len(other) == 0 and len(bad_pairs) == 0 and n_issues == 0 and len(dupes) == 0:
    print('✅ All validation checks passed.')
else:
    print('⚠️ Some issues found — review above.')


DATA QUALITY SUMMARY
Date range:        2020-01-01 → 2026-09-30
Outright contracts: 29
Spread instruments: 55
Roll pairs:         28
OHLCV rows:         7,916
OI rows:            5,334
Panel rows:         837
Crossovers found:   27
Symbol issues:      753
Pair seq issues:    0
FND issues:         0
OHLCV duplicates:   0
⚠️ Some issues found — review above.
